# AireIQ · S4: regresión de humedad con MLlib
**Responsable: Eberth Emmanuel Rosales Castillo.**

Pregunta U1: ¿Qué tan precisamente se estima la humedad relativa usando temperatura, presión y hora del mismo instante?

Se utiliza Gold de S3, con una observación por minuto. Esta estimación contemporánea no es el pronóstico a un minuto de U2 y no demuestra causalidad.

Método: primer 60 % cronológico para entrenar, siguiente 20 % para seleccionar por RMSE y último 20 % para prueba final. Se preservan timestamps completos en cada corte. La separación temporal es una decisión propia frente al ejemplo aleatorio de U1: reduce la mezcla de mediciones vecinas entre entrenamiento y prueba. La prueba no elige hiperparámetros. Se vuelve a ajustar la configuración ganadora con el primer 80 % y se evalúa una vez en el último 20 %.

Se comparan LinearRegression, Ridge, Elastic Net y RandomForestRegressor, más una referencia constante (media de entrenamiento). RMSE y MAE se expresan en puntos porcentuales de humedad; R² puede ser negativo. No se presupone que los modelos superen la referencia.

Referencia: https://262bigdata.github.io/lambda26/proyecto-sello/u1-producto/


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder.appName("AireIQ-S4-Humedad").master("local[2]")
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.sql.shuffle.partitions", "4").getOrCreate())
# Elegir solo ejecuciones S3 terminadas, con manifiesto y Parquet.
root = Path("/opt/aireiq/outputs")
sources = sorted(p for p in (root / "s03").iterdir() if (p / "manifest.json").is_file() and (p / "gold/_SUCCESS").is_file())
assert sources, "Ejecutar S3 primero"
source = sources[-1]
s03 = json.loads((source / "manifest.json").read_text())
assert s03["roundtrip_valores"] and s03["partition_filters"]
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = root / "s04" / run_id
out.mkdir(parents=True, exist_ok=False)
data = spark.read.parquet(str(source / "gold")).cache()
times = [r.minuto for r in data.select("minuto").distinct().orderBy("minuto").collect()]
assert len(times) >= 100, "Historial insuficiente para tres bloques"
cut1, cut2 = times[int(len(times)*.6)], times[int(len(times)*.8)]
train = data.filter(F.col("minuto") < F.lit(cut1)).cache()
validation = data.filter((F.col("minuto") >= F.lit(cut1)) & (F.col("minuto") < F.lit(cut2))).cache()
test = data.filter(F.col("minuto") >= F.lit(cut2)).cache()
counts = {"train": train.count(), "validation": validation.count(), "test": test.count()}
assert min(counts.values()) > 0 and sum(counts.values()) == data.count()
for left, right in [(train, validation), (validation, test)]:
    assert left.agg(F.max("minuto")).first()[0] < right.agg(F.min("minuto")).first()[0]
print("S3:", source.name, "Filas por bloque:", counts)
# Mostrar cortes como strings en zona de Lima; internamente los timestamps son instantes UTC.
for label, frame in [("train", train), ("validation", validation), ("test", test)]:
    print(label, frame.select(F.min("minuto").cast("string"), F.max("minuto").cast("string")).first())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 03:01:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 03:01:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/20 03:01:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
                                                                                

S3: 20261120T025546040810Z Filas por bloque: {'train': 2535, 'validation': 845, 'test': 845}
train Row(CAST(min(minuto) AS STRING)='2026-10-21 23:52:00', CAST(max(minuto) AS STRING)='2026-11-06 02:57:00')
validation Row(CAST(min(minuto) AS STRING)='2026-11-06 02:58:00', CAST(max(minuto) AS STRING)='2026-11-18 06:16:00')
test Row(CAST(min(minuto) AS STRING)='2026-11-18 06:17:00', CAST(max(minuto) AS STRING)='2026-11-19 20:50:00')


## Preparación y comparación
La hora se representa con seno y coseno, de modo que 23:59 y 00:00 sean cercanas. Estas transformaciones y VectorAssembler forman parte del modelo guardado. No se usan humedad actual ni índices derivados como predictores del objetivo contemporáneo.


In [2]:
features = ["temperatura_c", "presion_hpa", "hora_sin", "hora_cos"]
sql = "SELECT *, sin(2 * PI() * (hour(minuto) + minute(minuto)/60.0) / 24) AS hora_sin, cos(2 * PI() * (hour(minuto) + minute(minuto)/60.0) / 24) AS hora_cos FROM __THIS__"
def pipeline(name):
    regressors = {
        "lineal": LinearRegression(labelCol="humedad_pct", regParam=0.0, maxIter=100),
        "ridge": LinearRegression(labelCol="humedad_pct", regParam=0.1, elasticNetParam=0.0, maxIter=100),
        "elastic_net": LinearRegression(labelCol="humedad_pct", regParam=0.1, elasticNetParam=0.5, maxIter=100),
        "random_forest": RandomForestRegressor(labelCol="humedad_pct", numTrees=50, maxDepth=6, seed=42)
    }
    return Pipeline(stages=[SQLTransformer(statement=sql), VectorAssembler(inputCols=features, outputCol="features", handleInvalid="error"), regressors[name]])
def metrics(pred):
    result = {m: RegressionEvaluator(labelCol="humedad_pct", predictionCol="prediction", metricName=m).evaluate(pred) for m in ["rmse", "mae", "r2"]}
    assert all(math.isfinite(v) for v in result.values()), result
    return result

mean_train = train.agg(F.avg("humedad_pct")).first()[0]
baseline_val = metrics(validation.withColumn("prediction", F.lit(mean_train)))
results = []
for name in ["lineal", "ridge", "elastic_net", "random_forest"]:
    fitted = pipeline(name).fit(train)
    pred = fitted.transform(validation).cache()
    score = metrics(pred)
    results.append({"modelo": name, **score})
    pred.unpersist()
    print(name, score, flush=True)
winner = min(results, key=lambda r: (r["rmse"], r["modelo"]))["modelo"]
spark.createDataFrame(results + [{"modelo":"referencia_media", **baseline_val}]).select("modelo", "rmse", "mae", "r2").orderBy("rmse").show(truncate=False)
print("Mejor configuración ML en validación:", winner)


26/11/20 03:01:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/20 03:01:43 WARN Instrumentation: [6427f29c] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


lineal {'rmse': 5.667646023891026, 'mae': 4.648312388556594, 'r2': 0.35746933657961477}
ridge {'rmse': 5.71078635955574, 'mae': 4.616513301405425, 'r2': 0.34765062790210133}
elastic_net {'rmse': 5.736864846366921, 'mae': 4.586194152304845, 'r2': 0.34167907617839455}
random_forest {'rmse': 4.329596766105983, 'mae': 2.9952414602934816, 'r2': 0.6250413967543406}


[Stage 107:>                                                        (0 + 2) / 2]

+----------------+-----------------+------------------+--------------------+
|modelo          |rmse             |mae               |r2                  |
+----------------+-----------------+------------------+--------------------+
|random_forest   |4.329596766105983|2.9952414602934816|0.6250413967543406  |
|lineal          |5.667646023891026|4.648312388556594 |0.35746933657961477 |
|ridge           |5.71078635955574 |4.616513301405425 |0.34765062790210133 |
|elastic_net     |5.736864846366921|4.586194152304845 |0.34167907617839455 |
|referencia_media|7.354722419318235|5.277439542714421 |-0.08198438778585349|
+----------------+-----------------+------------------+--------------------+

Mejor configuración ML en validación: random_forest


In [3]:
development = train.unionByName(validation).cache()
final_model = pipeline(winner).fit(development)
pred_test = final_model.transform(test).cache()
test_metrics = metrics(pred_test)
mean_dev = development.agg(F.avg("humedad_pct")).first()[0]
baseline_test = metrics(test.withColumn("prediction", F.lit(mean_dev)))
final_model.write().save(str(out / "modelo_humedad_u1"))
loaded = PipelineModel.load(str(out / "modelo_humedad_u1"))
reloaded = loaded.transform(test).select("dispositivo", "minuto", F.col("prediction").alias("restored"))
delta = pred_test.select("dispositivo", "minuto", "prediction").join(reloaded, ["dispositivo", "minuto"]).select(F.max(F.abs(F.col("prediction")-F.col("restored")))).first()[0]
assert delta is not None and delta <= 1e-9, delta
assert pred_test.count() == test.count()
assert pred_test.filter(F.col("prediction").isNull() | F.isnan("prediction")).count() == 0
pred_test.select("dispositivo", "minuto", "fecha", "humedad_pct", "prediction").write.partitionBy("fecha").parquet(str(out / "predicciones_test"))
print("Prueba final:", test_metrics)
print("Referencia media:", baseline_test)
print("Supera referencia en RMSE:", test_metrics["rmse"] < baseline_test["rmse"])
print("Máxima diferencia tras recargar:", delta)


Prueba final: {'rmse': 4.098491478587124, 'mae': 3.2174829868051806, 'r2': -2.6561100188092888}
Referencia media: {'rmse': 3.211229321899784, 'mae': 2.4930445803901726, 'r2': -1.244470101142479}
Supera referencia en RMSE: False
Máxima diferencia tras recargar: 0.0


## Indicador descriptivo
Referencia estadística: percentiles p5/p95 de los minutos del primer 60 % del historial, calculados solo con entrenamiento. Este periodo es una decisión metodológica explícita, pendiente de aprobación académica; no se reutilizan los umbrales de IAQ del brief. El indicador solicitado se calcula sobre **lecturas válidas Silver**, con denominador por día/hora. Se entrega además una vista por minuto, que reduce el sesgo de frecuencia. Un porcentaje de lecturas no equivale a un porcentaje de tiempo.


In [4]:
p5, p95 = train.approxQuantile("humedad_pct", [.05, .95], 0.0)
def describe(frame, timestamp, unit, keys):
    tagged = (frame.withColumn("fecha", F.to_date(timestamp)).withColumn("hora_lima", F.hour(timestamp))
        .withColumn("fuera", ((F.col("humedad_pct") < p5) | (F.col("humedad_pct") > p95)).cast("int")))
    return (tagged.groupBy(*keys).agg(F.count("*").alias(unit), F.avg("humedad_pct").alias("humedad_media_pct"),
        (100*F.avg("fuera")).alias("porcentaje_fuera_referencia")))
silver = spark.read.parquet(str(source / "silver"))
daily = describe(silver, "created_at", "lecturas_validas", ["fecha"])
hourly = describe(silver, "created_at", "lecturas_validas", ["hora_lima"])
minute_summary = describe(data, "minuto", "minutos_observados", ["fecha"])
daily.write.parquet(str(out / "descriptivo_diario_lecturas"))
hourly.write.parquet(str(out / "descriptivo_horario_lecturas"))
minute_summary.write.parquet(str(out / "descriptivo_diario_minutos"))
print("Referencia humedad (%):", {"p5":p5, "p95":p95})
daily.orderBy("fecha").show(40, truncate=False)
hourly.orderBy("hora_lima").show(24, truncate=False)
report = {"run_id": run_id, "s03_run": source.name, "source_sha256": s03["source_sha256"],
    "spark_version": spark.version, "timezone": "America/Lima", "objetivo": "humedad contemporanea media por minuto",
    "split": "cronologico 60/20/20", "conteos": counts, "features": features,
    "validacion": results, "baseline_validacion": baseline_val, "modelo_elegido": winner,
    "test": test_metrics, "baseline_test": baseline_test,
    "supera_baseline_test": test_metrics["rmse"] < baseline_test["rmse"],
    "diferencia_recarga": delta, "p5": p5, "p95": p95,
    "referencia": "primer 60% de minutos observados; no umbral sanitario",
    "fuera_rango_predicciones_test": pred_test.filter(~F.col("prediction").between(0,100)).count()}
(out / "manifest.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Evidencias:", out)
for frame in [data, train, validation, test, development, pred_test]: frame.unpersist()


Referencia humedad (%): {'p5': 30.283099999999997, 'p95': 42.969674999999995}


+----------+----------------+------------------+---------------------------+
|fecha     |lecturas_validas|humedad_media_pct |porcentaje_fuera_referencia|
+----------+----------------+------------------+---------------------------+
|2026-10-21|5               |37.265660000000004|0.0                        |
|2026-10-22|771             |39.45005343709464 |0.38910505836575876        |
|2026-10-23|10              |39.5558           |0.0                        |
|2026-10-28|195             |35.760983589743596|0.0                        |
|2026-10-29|2255            |34.54182700665187 |4.878048780487805          |
|2026-10-30|687             |40.091847889374094|66.95778748180496          |
|2026-10-31|145             |31.591694482758626|27.586206896551722         |
|2026-11-01|1411            |32.64578596739898 |11.76470588235294          |
|2026-11-05|378             |36.44720502645502 |0.0                        |
|2026-11-06|1825            |42.698310575342504|55.83561643835616          |

[Stage 190:>                                                        (0 + 2) / 2]

+---------+----------------+------------------+---------------------------+
|hora_lima|lecturas_validas|humedad_media_pct |porcentaje_fuera_referencia|
+---------+----------------+------------------+---------------------------+
|0        |584             |36.875035616438375|0.0                        |
|1        |535             |38.27341401869165 |0.0                        |
|2        |529             |39.57659376181475 |0.0                        |
|3        |592             |39.43710016891897 |37.16216216216216          |
|4        |397             |40.46016120906802 |26.700251889168765         |
|5        |455             |41.24385230769232 |35.38461538461539          |
|6        |367             |37.736055858310614|7.084468664850137          |
|7        |291             |34.690074226804136|0.3436426116838488         |
|8        |389             |35.17879691516709 |19.28020565552699          |
|9        |716             |42.6144675977654  |68.29608938547486          |
|10       |8

## Interpretación para sustentación
- RMSE penaliza más los errores grandes; MAE expresa el error absoluto promedio.
- R² negativo indica peor ajuste que predecir la media del conjunto evaluado. La referencia operativa usa la media del entrenamiento, no la del test.
- El modelo elegido es el mejor ML de la comparación en validación; su utilidad depende de la prueba final y de la comparación con la referencia. Guardarlo no demuestra que sea útil en producción.
- Los huecos y el único sensor limitan la generalización. No se modifican métricas ni se cambia el ganador mirando el test.
- U2 requerirá otro objetivo: humedad de t+1 minuto, pares temporales correctos y comparación con persistencia. Este cuaderno no implementa esa inferencia.
